# photonviz — quickstart

GPU-accelerated (WebGL2) charts inside the notebook. Arrays cross to the
browser as binary buffers, so these plots stay interactive — scroll to zoom,
drag to pan, hover for a tooltip.

```bash
pip install photonviz
```

> **Google Colab:** run `from google.colab import output; output.enable_custom_widget_manager()` once per session.

In [1]:
import numpy as np
import photonviz as pv

# 200k points — pan and zoom stay at 60fps.
x = np.linspace(0, 40, 200_000)
y = np.sin(x) + 0.3 * np.sin(x * 7) + 0.05 * np.random.default_rng(0).standard_normal(x.size)

(pv.Plot(theme="dark", title="200,000 points", legend=True, height="320px")
   .line(x, y, color="#60a5fa", width=1.5, name="signal")
   .hline(0, color="#64748b", dash=[4, 4]))

ModuleNotFoundError: No module named 'photonviz'

In [2]:
# Bubble chart + a least-squares fit with a confidence band.
rng = np.random.default_rng(7)
n = 220
gx = rng.normal(0, 1.1, n)
gy = 0.8 * gx + rng.normal(0, 0.6, n)

(pv.Plot(theme="dark", title="Bubbles + OLS fit", legend=True, pick="xy", height="320px")
   .scatter(gx, gy, sizes=4 + np.abs(rng.normal(0, 1, n)) * 14, color="#38bdf8", name="samples")
   .regression(gx, gy, band=2, color="#f472b6"))

In [3]:
# A confusion matrix and a correlation heatmap both get a colorbar automatically.
rng = np.random.default_rng(3)
true = rng.integers(0, 5, 600)
pred = np.where(rng.random(600) < 0.82, true, rng.integers(0, 5, 600))

pv.confusion_matrix(true, pred, classes=5, colormap="viridis",
                    plot={"theme": "dark", "title": "Confusion matrix", "height": "340px"})

In [4]:
# Model architecture in 3D: one cuboid per layer, sized from its output tensor.
# `pv.model_graph_3d(torch_model, example_input=...)` does this straight from a
# PyTorch / Keras / scikit-learn / ONNX model; here is the same graph by hand.
cnn = pv.from_layers([
    {"id": "input",  "name": "input",  "type": "Input",             "shape": [3, 224, 224]},
    {"id": "conv1",  "name": "conv1",  "type": "Conv2d",            "shape": [64, 112, 112], "params": 1792},
    {"id": "pool1",  "name": "pool1",  "type": "MaxPool2d",         "shape": [64, 56, 56]},
    {"id": "conv2",  "name": "conv2",  "type": "Conv2d",            "shape": [128, 56, 56], "params": 73856},
    {"id": "pool2",  "name": "pool2",  "type": "MaxPool2d",         "shape": [128, 28, 28]},
    {"id": "conv3",  "name": "conv3",  "type": "Conv2d",            "shape": [256, 28, 28], "params": 295168},
    {"id": "gap",    "name": "gap",    "type": "AdaptiveAvgPool2d", "shape": [256, 1, 1]},
    {"id": "fc",     "name": "fc",     "type": "Linear",            "shape": [1000], "params": 257000},
], name="TinyVGG")

pv.model_graph_3d(
    cnn, labels="full", rankSpacing=0.45,
    plot={"aspectMode": "data", "projection": "orthographic", "showAxes": False,
          "gridPlanes": False, "azimuth": 0.45, "elevation": 0.3, "distance": 0.95,
          "height": "360px"},
)

In [5]:
# A lit 3D surface — drag to orbit, wheel to zoom.
cols = rows = 64
u = np.linspace(-3, 3, cols)[None, :]
v = np.linspace(-3, 3, rows)[:, None]
r = np.hypot(u, v)
z = np.sin(r * 2.2) / (r + 0.6)

pv.surface(z.ravel(), cols, rows, colormap="turbo",
           plot={"title": "sinc", "axisLabels": {"x": "x", "y": "z", "z": "y"}, "height": "360px"})